# 第6章 文件 I/O、FITS 与数据格式

这个 notebook 用最小教学数据演示三件事：相对路径读取、CSV 结构化读入，以及“数据 + 元数据”必须一起看待的 FITS 思维。因为当前环境不依赖 `astropy`，这里用一个教学版 header 字典代替真实 FITS header。

## 学习目标

- 理解文本、CSV 和 FITS 思维的差异
- 学会使用相对路径读取教学数据
- 建立“数据和元数据必须一起读取”的意识
- 在没有专门天文库时也能先练习 I/O 骨架

In [ ]:
from __future__ import annotations

import csv
import platform
from pathlib import Path

DATA_PATH = Path('../../data/small/planetary_orbits_demo.csv')

print('data exists:', DATA_PATH.exists())
print('relative path:', DATA_PATH)
print('absolute path:', DATA_PATH.resolve())
print('Python version:', platform.python_version())


## 先看原始文本视角

文件 I/O 的第一层往往不是“直接分析”，而是先确认文件到底长什么样。

In [ ]:
raw_text = DATA_PATH.read_text(encoding='utf-8')
raw_lines = raw_text.strip().splitlines()
print('line count:', len(raw_lines))
print('header line:', raw_lines[0])
print('first two data lines:')
for line in raw_lines[1:3]:
    print('  ', line)


## 用 CSV 方式结构化读入

文本文件只是字节序列；CSV 读入的价值在于把每一列重新恢复成有名字的字段。

In [ ]:
with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    for key in ('semi_major_axis_au', 'orbital_period_years', 'mass_earth', 'radius_earth'):
        row[key] = float(row[key])

print('row count:', len(rows))
print('first row:', rows[0])
print('last row:', rows[-1])


## 字段契约：先确认表格结构

读入 CSV 之后，不要马上使用字段。先写清楚这张表至少应该有哪些列、哪些列应该能转成数值。这个最小“字段契约”可以把很多错误挡在工作流入口。

In [ ]:
required_fields = {
    'planet',
    'semi_major_axis_au',
    'orbital_period_years',
    'mass_earth',
    'radius_earth',
}

available_fields = set(rows[0].keys())
missing_fields = required_fields - available_fields

print('available fields:', sorted(available_fields))
print('missing required fields:', sorted(missing_fields))
assert not missing_fields


## 缺失值和类型转换

教学数据很干净，但真实 CSV 常常会出现空字符串、`NA` 或无法转成数字的字段。下面的函数把“缺失”和“非法数值”显式区分开。

In [ ]:
def parse_float(value: str, field_name: str) -> float | None:
    text = value.strip()
    if text in {'', 'NA', 'nan', 'None'}:
        return None
    try:
        return float(text)
    except ValueError as exc:
        message = f'{field_name} is not numeric: {value!r}'
        raise ValueError(message) from exc

demo_values = ['1.25', 'NA', 'not-a-number']
for value in demo_values[:2]:
    print(value, '->', parse_float(value, 'demo_field'))

try:
    parse_float(demo_values[2], 'demo_field')
except ValueError as exc:
    print('caught expected parsing issue:', exc)


## 一个最小数据检查

I/O 之后的第一件事，通常不是马上建模，而是先确认量级、字段和单位是否合理。

In [ ]:
semi_major_axes = [row['semi_major_axis_au'] for row in rows]
orbital_periods = [row['orbital_period_years'] for row in rows]

print('semi-major axis range [au]:', (min(semi_major_axes), max(semi_major_axes)))
print('orbital period range [yr]:', (min(orbital_periods), max(orbital_periods)))
print('planet names:', [row['planet'] for row in rows])


## provenance：记录数据从哪里来

数据能读入还不够，还要记录来源、字段单位和关键假设。这个小字典不是分析结果，却是复现分析时非常有用的线索。

In [ ]:
data_provenance = {
    'source_file': str(DATA_PATH),
    'format': 'csv',
    'required_fields': sorted(required_fields),
    'unit_notes': {
        'semi_major_axis_au': 'au',
        'orbital_period_years': 'yr',
        'mass_earth': 'Earth mass',
        'radius_earth': 'Earth radius',
    },
}

for key, value in data_provenance.items():
    print(f'{key}: {value}')


## FITS 思维：为什么还需要元数据

CSV 里通常只有表格本体；而在真实天文中，FITS 更重要的一点，是它会把“数据是什么”与“这些数据怎么解释”放在一起。

In [ ]:
teaching_header = {
    'OBJECT': 'SolarSystemTeachingTable',
    'BUNIT': 'mixed physical units by column',
    'ORIGIN': 'AIforAstronomers educational repository',
    'COMMENT': 'In a real FITS file, header keywords travel with the data.',
}

print('teaching header keys:', sorted(teaching_header))
for key, value in teaching_header.items():
    print(f'  {key}: {value}')


## 数据和元数据必须一起解释

下面这个最小快照展示了为什么光有数值列不够：你还需要知道这些列代表什么、单位是什么、来源是什么。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'columns': list(rows[0].keys()),
    'field_contract_ok': not missing_fields,
    'object_keyword': teaching_header['OBJECT'],
    'origin_keyword': teaching_header['ORIGIN'],
}

print('I/O snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')


## 小结

这一章最重要的不是会不会某个库函数，而是建立一条朴素但可靠的 I/O 骨架：先确认路径，再看原始文本，再结构化读入，随后检查字段与量级，最后把元数据一起纳入解释。这正是后面进入 FITS 图像和光谱文件时最需要的思维。